# PyToch: Multi Class Dataset Classification

In [ ]:
import torch
from torch import nn
from torch import optim
from torchmetrics import Accuracy
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
print(f'PyTorch: version {torch.__version__}')
print(f'PyTorch: {device.type.upper()} device (index: {device.index})')

In [ ]:
N_SAMPLES = 1000
N_CLASSES = 4
N_FEATURES = 2
THRESHOLD = 0.6

## Prepare Datasets

In [ ]:
x_blob, y_blob = make_blobs(n_samples=N_SAMPLES,
                            n_features=N_FEATURES,
                            centers=N_CLASSES,
                            cluster_std=0.75,
                            random_state=0)

In [ ]:
x_blob = torch.from_numpy(x_blob).type(torch.float)
y_blob = torch.from_numpy(y_blob).type(torch.long)

In [ ]:
# Split data into train and test datasets
x_train, x_test, y_train, y_test = train_test_split(x_blob, y_blob,
                                                    test_size=0.2,
                                                    random_state=0)

# Upload datasets to device
x_train = x_train.to(device)
y_train = y_train.to(device)
x_test = x_test.to(device)
y_test = y_test.to(device)

In [ ]:
plt.figure(figsize=(10, 10))
plt.scatter(x_blob[:, 0], x_blob[:, 1], c=y_blob, cmap='RdYlBu');

## Define Model

In [ ]:
class BlobModel(nn.Module):
    def __init__(self,
                 in_features: int = N_FEATURES,
                 out_features: int = N_CLASSES,
                 hidden_units: int = 8):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, hidden_units),
            nn.ReLU(),
            nn.Linear(hidden_units, out_features)
        )

    def forward(self, x):
        y = self.layers(x)
        return y

In [ ]:
model = BlobModel().to(device)

## Training

In [ ]:
# Create a loss function
loss_fn = nn.CrossEntropyLoss()

In [ ]:
# Create a metric function
metric_fn = Accuracy(task='multiclass', num_classes=N_CLASSES).to(device)

### Training Loop

In [ ]:
N_EPOCHS = 100

tr_losses = []
ts_losses = []
tr_accs = []
ts_accs = []

# Create an optimizer
optimizer = optim.SGD(model.parameters(),
                      momentum=0.9,
                      lr=0.1)

for epoch in range(N_EPOCHS):
    ## Training step
    model.train()
    y_logits = model(x_train)
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
    tr_loss = loss_fn(y_logits, y_train)
    tr_accu = metric_fn(y_pred, y_train)
    optimizer.zero_grad()
    tr_loss.backward()
    optimizer.step()

    ## Testing step
    model.eval()
    with torch.inference_mode():
        y_logits = model(x_test)
        y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
        ts_loss = loss_fn(y_logits, y_test)
        ts_accu = metric_fn(y_pred, y_test)

    if epoch % 10 == 0:
        tr_losses.append(tr_loss.item())
        ts_losses.append(ts_loss.item())
        tr_accs.append(tr_accu)
        ts_accs.append(ts_accu)
        print(f'Epoch: {epoch:02} | L/A: {tr_loss:.3f}/{tr_accu:3.1f} | Test L/A: {ts_loss:.3f}/{ts_accu:3.1f}')

## Evaluating

In [ ]:
model.eval()
with torch.inference_mode():
    y_logits = model(x_test)
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)

loss = loss_fn(y_logits, y_test)
print(f'Final Loss: {loss:.3f}')